In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv(
    "../data/processed/cleaned_deliveries.csv",
    dtype={"ball": str, "match_id": str},
    low_memory=False,
)
matches = pd.read_csv("../data/processed/matches.csv", dtype={"match_id": str})

# --- Batter-level helper flags ---
df["ball_faced"] = df["wides"].isna().astype(int)

df["is_boundary"] = (
    df["runs_off_bat"].isin([4, 6])
    & (~df["non_boundary"].fillna(False).astype(bool))
).astype(int)

df["is_dot_for_batter"] = (
    df["is_legal"] & (df["runs_off_bat"] == 0)
).astype(int)

df["is_batter_dismissal"] = (
    df["is_wicket"] & (df["player_dismissed"] == df["striker"])
).astype(int)

print("Rows:", len(df))
print("Batters (any appearance):", df["striker"].nunique())
print("Total balls faced (all batters):", int(df["ball_faced"].sum()))
print("Total batter dismissals:", int(df["is_batter_dismissal"].sum()))

Rows: 295557
Batters (any appearance): 739
Total balls faced (all batters): 285686
Total batter dismissals: 14088


In [4]:
# Melt into long format: for each delivery, both striker and non_striker "appear"
melted = pd.concat([
    df[["match_id", "innings", "over", "ball_in_over", "striker"]]
        .assign(role=0).rename(columns={"striker": "batter"}),
    df[["match_id", "innings", "over", "ball_in_over", "non_striker"]]
        .assign(role=1).rename(columns={"non_striker": "batter"}),
], ignore_index=True)

melted = melted.sort_values(
    ["match_id", "innings", "over", "ball_in_over", "role"]
)

first_app = melted.drop_duplicates(["match_id", "innings", "batter"]).copy()
first_app["batting_position"] = first_app.groupby(["match_id", "innings"]).cumcount() + 1

pos_map = first_app[["match_id", "innings", "batter", "batting_position"]].rename(
    columns={"batter": "striker"}
)
df = df.merge(pos_map, on=["match_id", "innings", "striker"], how="left")

print("Batting positions assigned. Distribution:")
print(df["batting_position"].value_counts().sort_index().to_string())


Batting positions assigned. Distribution:
batting_position
1     54276
2     55726
3     49411
4     44073
5     35072
6     24087
7     15700
8      8814
9      4785
10     2538
11     1075


In [5]:
MIN_BALLS_CAREER = 300

career = df.groupby("striker").agg(
    balls=("ball_faced", "sum"),
    runs=("runs_off_bat", "sum"),
    dismissals=("is_batter_dismissal", "sum"),
    boundaries=("is_boundary", "sum"),
    dot_balls=("is_dot_for_batter", "sum"),
)

career["innings"] = (
    df[["striker", "match_id", "innings"]]
    .drop_duplicates()
    .groupby("striker").size()
)

career["average"] = np.where(career["dismissals"] > 0,
                              career["runs"] / career["dismissals"], np.nan)
career["strike_rate"] = (career["runs"] / career["balls"] * 100).round(2)
career["boundary_pct"] = (career["boundaries"] / career["balls"] * 100).round(2)
career["dot_ball_pct"] = (career["dot_balls"] / career["balls"] * 100).round(2)
career["runs_per_innings"] = (career["runs"] / career["innings"]).round(2)

career = career.reset_index().query("balls >= @MIN_BALLS_CAREER").copy()

print(f"Qualified batters (>= {MIN_BALLS_CAREER} balls):", len(career))
print("\nTop 10 by total runs:")
print(
    career.nlargest(10, "runs")[
        ["striker", "innings", "runs", "balls", "average", "strike_rate",
         "boundary_pct", "dot_ball_pct"]
    ].round(2).to_string(index=False)
)


Qualified batters (>= 300 balls): 186

Top 10 by total runs:
       striker  innings  runs  balls  average  strike_rate  boundary_pct  dot_ball_pct
       V Kohli      275  9336   6926    40.77       134.80         16.75         32.49
     RG Sharma      275  7329   5514    30.16       132.92         17.85         37.74
      S Dhawan      221  6769   5324    36.39       127.14         17.28         36.96
     DA Warner      184  6565   4697    42.08       139.77         19.14         36.41
      KL Rahul      149  5815   4179    47.28       139.15         17.88         32.76
      SK Raina      200  5528   4043    34.34       136.73         17.54         32.97
      MS Dhoni      241  5439   3957    39.13       137.45         16.15         33.54
     AM Rahane      197  5367   4273    31.20       125.60         15.91         36.37
     SV Samson      185  5181   3671    32.79       141.13         18.39         34.27
AB de Villiers      170  5162   3403    42.31       151.69         19

In [6]:
phase_agg = df.groupby(["striker", "phase"]).agg(
    runs=("runs_off_bat", "sum"),
    balls=("ball_faced", "sum"),
    dismissals=("is_batter_dismissal", "sum"),
).reset_index()

phase_wide = phase_agg.pivot(
    index="striker", columns="phase",
    values=["runs", "balls", "dismissals"]
)

# Flatten: 'runs_Powerplay', 'balls_Death', etc.
phase_wide.columns = [f"{stat}_{phase}" for stat, phase in phase_wide.columns]
phase_wide = phase_wide.reset_index().fillna(0)

for phase in ["Powerplay", "Middle", "Death"]:
    runs_col  = f"runs_{phase}"
    balls_col = f"balls_{phase}"
    dism_col  = f"dismissals_{phase}"
    phase_wide[f"SR_{phase}"] = np.where(
        phase_wide[balls_col] > 0,
        (phase_wide[runs_col] / phase_wide[balls_col] * 100).round(1),
        np.nan
    )
    phase_wide[f"avg_{phase}"] = np.where(
        phase_wide[dism_col] > 0,
        (phase_wide[runs_col] / phase_wide[dism_col]).round(1),
        np.nan
    )

print("Phase stats built for", len(phase_wide), "batters")
print("\nSample (first 3 batters):")
cols = ["striker",
        "balls_Powerplay", "SR_Powerplay",
        "balls_Middle",    "SR_Middle",
        "balls_Death",     "SR_Death"]
print(phase_wide[cols].head(3).to_string())

Phase stats built for 739 batters

Sample (first 3 batters):
          striker  balls_Powerplay  SR_Powerplay  balls_Middle  SR_Middle  balls_Death  SR_Death
0  A Ashish Reddy              6.0          83.3          41.0      148.8        146.0     146.6
1        A Badoni             99.0         125.3         410.0      125.9        327.0     164.5
2      A Chandila              0.0           NaN           0.0        NaN          7.0      57.1


In [7]:
phase_agg = df.groupby(["striker", "phase"]).agg(
    runs=("runs_off_bat", "sum"),
    balls=("ball_faced", "sum"),
    dismissals=("is_batter_dismissal", "sum"),
).reset_index()

phase_wide = phase_agg.pivot(
    index="striker", columns="phase",
    values=["runs", "balls", "dismissals"]
)

# Flatten: 'runs_Powerplay', 'balls_Death', etc.
phase_wide.columns = [f"{stat}_{phase}" for stat, phase in phase_wide.columns]
phase_wide = phase_wide.reset_index().fillna(0)

for phase in ["Powerplay", "Middle", "Death"]:
    runs_col  = f"runs_{phase}"
    balls_col = f"balls_{phase}"
    dism_col  = f"dismissals_{phase}"
    phase_wide[f"SR_{phase}"] = np.where(
        phase_wide[balls_col] > 0,
        (phase_wide[runs_col] / phase_wide[balls_col] * 100).round(1),
        np.nan
    )
    phase_wide[f"avg_{phase}"] = np.where(
        phase_wide[dism_col] > 0,
        (phase_wide[runs_col] / phase_wide[dism_col]).round(1),
        np.nan
    )

print("Phase stats built for", len(phase_wide), "batters")
print("\nSample (first 3 batters):")
cols = ["striker",
        "balls_Powerplay", "SR_Powerplay",
        "balls_Middle",    "SR_Middle",
        "balls_Death",     "SR_Death"]
print(phase_wide[cols].head(3).to_string())

Phase stats built for 739 batters

Sample (first 3 batters):
          striker  balls_Powerplay  SR_Powerplay  balls_Middle  SR_Middle  balls_Death  SR_Death
0  A Ashish Reddy              6.0          83.3          41.0      148.8        146.0     146.6
1        A Badoni             99.0         125.3         410.0      125.9        327.0     164.5
2      A Chandila              0.0           NaN           0.0        NaN          7.0      57.1


In [8]:
innings_runs = (
    df.groupby(["striker", "match_id", "innings"])
      .agg(runs=("runs_off_bat", "sum"), balls=("ball_faced", "sum"))
      .reset_index()
)

consistency = innings_runs.groupby("striker").agg(
    innings_batted=("match_id", "count"),
    mean_innings=("runs", "mean"),
    std_innings=("runs", "std"),
    best_innings=("runs", "max"),
    fifties=("runs", lambda x: (x >= 50).sum()),
    ducks=("runs", lambda x: (x == 0).sum()),
).round(2).reset_index()

consistency["fifty_plus_pct"] = (
    consistency["fifties"] / consistency["innings_batted"] * 100
).round(1)
consistency["duck_pct"] = (
    consistency["ducks"] / consistency["innings_batted"] * 100
).round(1)

print("Consistency metrics built for", len(consistency), "batters")
print("\nSample (first 3):")
print(consistency.head(3).to_string(index=False))

Consistency metrics built for 739 batters

Sample (first 3):
       striker  innings_batted  mean_innings  std_innings  best_innings  fifties  ducks  fifty_plus_pct  duck_pct
A Ashish Reddy              23         12.17         8.88            36        0      0             0.0       0.0
      A Badoni              56         21.04        18.59            74        7      5            12.5       8.9
    A Chandila               2          2.00         2.83             4        0      1             0.0      50.0


In [9]:
batting = (
    career
    .merge(phase_wide, on="striker", how="left")
    .merge(consistency, on="striker", how="left")
)

# Batting position per batter
batter_positions = (
    df[["match_id", "innings", "striker", "batting_position"]]
    .drop_duplicates(["match_id", "innings", "striker"])
)

opener_share = (
    batter_positions.groupby("striker")["batting_position"]
    .apply(lambda x: (x <= 2).mean())
    .rename("opener_innings_pct")
    .reset_index()
)

modal_pos = (
    batter_positions.groupby("striker")["batting_position"]
    .agg(lambda x: x.mode().iloc[0] if len(x.mode()) else np.nan)
    .rename("modal_position")
    .reset_index()
)

batting = batting.merge(opener_share, on="striker", how="left")
batting = batting.merge(modal_pos, on="striker", how="left")

def classify_role(row):
    if pd.notna(row["opener_innings_pct"]) and row["opener_innings_pct"] >= 0.5:
        return "Opener"
    if (row.get("balls_Death", 0) >= 100
        and row.get("SR_Death", 0) >= 130
        and row.get("balls_Death", 0) >= row.get("balls_Middle", 0)):
        return "Finisher"
    if (row.get("balls_Middle", 0) >= 100
        and row.get("avg_Middle", 0) >= 30
        and row.get("SR_Middle", 100) < 140):
        return "Anchor"
    return "Middle-order"

batting["role"] = batting.apply(classify_role, axis=1)

print("Role distribution:")
print(batting["role"].value_counts().to_string())
print()

known = ["V Kohli", "DA Warner", "AB de Villiers", "MS Dhoni",
         "RG Sharma", "KL Rahul", "KA Pollard", "HH Pandya"]
sample = batting[batting["striker"].isin(known)][
    ["striker", "innings", "runs", "average", "strike_rate",
     "SR_Powerplay", "SR_Middle", "SR_Death", "role"]
].round(2)
print("Known players:")
print(sample.to_string(index=False))

Role distribution:
role
Middle-order    68
Opener          55
Anchor          44
Finisher        19

Known players:
       striker  innings  runs  average  strike_rate  SR_Powerplay  SR_Middle  SR_Death     role
AB de Villiers      170  5162    42.31       151.69         119.1      131.3     222.9   Anchor
     DA Warner      184  6565    42.08       139.77         138.3      134.8     176.9   Opener
     HH Pandya      150  2955    28.41       146.29          98.3      134.1     168.7   Anchor
    KA Pollard      168  3412    28.92       147.32          77.8      124.5     170.2 Finisher
      KL Rahul      149  5815    47.28       139.15         128.7      138.8     178.9   Opener
      MS Dhoni      241  5439    39.13       137.45          81.9      100.5     175.0 Finisher
     RG Sharma      275  7329    30.16       132.92         126.4      122.5     181.1   Anchor
       V Kohli      275  9336    40.77       134.80         129.5      125.8     188.4   Opener


In [10]:
if "role" not in batting.columns:
    raise RuntimeError("Cell 6 didn't run — the 'role' column is missing. "
                       "Go back and run Cell 6 first.")

batting.to_csv("../data/processed/batting_stats.csv", index=False)
print("Saved: data/processed/batting_stats.csv")
print("Shape:", batting.shape)
print()

print("=== Top 10 by strike rate (min 300 balls) ===")
print(
    batting.nlargest(10, "strike_rate")[
        ["striker", "innings", "runs", "balls", "strike_rate",
         "average", "boundary_pct", "role"]
    ].round(2).to_string(index=False)
)
print()

print("=== Top 10 by average (min 300 balls) ===")
print(
    batting.nlargest(10, "average")[
        ["striker", "innings", "runs", "average", "strike_rate",
         "SR_Death", "role"]
    ].round(2).to_string(index=False)
)
print()

print("=== Role distribution ===")
print(batting["role"].value_counts().to_string())

Saved: data/processed/batting_stats.csv
Shape: (186, 38)

=== Top 10 by strike rate (min 300 balls) ===
        striker  innings  runs  balls  strike_rate  average  boundary_pct         role
  V Suryavanshi       23  1028    449       228.95    44.70         39.42       Opener
  Priyansh Arya       31   909    471       192.99    29.32         31.85       Opener
       TH David       59  1151    650       177.08    33.85         24.46     Finisher
        PD Salt       40  1258    721       174.48    34.00         28.43       Opener
     AD Russell      114  2651   1522       174.18    29.79         26.87 Middle-order
      JP Inglis       16   544    314       173.25    38.86         27.39 Middle-order
Abhishek Sharma       89  2379   1389       171.27    29.74         26.49       Opener
        TM Head       52  1556    915       170.05    33.11         27.32       Opener
Ashutosh Sharma       26   564    334       168.86    29.68         23.65     Finisher
   RD Rickelton       26  